# Illustration of xQTL protocol

This notebook illustrates the computational protocols available from this repository for the detection and analysis of molecular QTLs (xQTLs). A minimal toy data-set consisting of 49 de-identified samples are used for the analysis.

The sections below provide a **step-by-step guide** for new users to set up the software environment, clone the protocol repository, and run their first xQTL analysis (for example, generating xQTLs with TensorQTL). These instructions are intended to be portable — they work on a personal Linux/macOS workstation as well as on any HPC cluster (SLURM, LSF, SGE, etc.). Nothing below is tied to a specific institution's cluster.

## Step-by-step getting started guide

The workflow below gets a new user from a fresh shell to running a pipeline in four steps:

1. **Install SoS** (Script of Scripts) in a conda environment — this is the workflow engine that drives the xQTL pipelines.
2. **Install the xQTL project's software stack with pixi** — this replaces the previous Singularity/Docker container workflow.
3. **Clone the `xqtl-protocol` repository**.
4. **Run an example pipeline** (e.g. TensorQTL) using the toy dataset.

### 1. Install SoS in a conda environment

Follow the official SoS instructions for a conda-based installation: <https://vatlab.github.io/sos-docs/running.html#Conda-installation>.

A typical minimal install looks like:

```bash
conda create -n sos -c conda-forge sos sos-pbs sos-bash sos-python sos-r sos-notebook jupyterlab-sos -y
conda activate sos
```

You only need to do this once per machine / user account.

### 2. Install the xQTL software stack with pixi

We no longer use Singularity/Docker containers as the default software environment. Instead, the xQTL protocol's dependencies (R, Python, TensorQTL, bcftools, plink, etc.) are installed through [pixi](https://pixi.sh/) using the StatFunGen `pixi-setup` installer. This gives you a self-contained, reproducible environment that works on any Linux/macOS system.

For the full, up-to-date installation guide (including options, troubleshooting, and advanced configuration), see the Wang Lab documentation:

- <https://wanggroup.org/hpc/docs/software-setup-conda> — *Advanced Software Setup with Pixi*
- <https://wanggroup.org/orientation/jupyter-setup.html> — *Software environment setup overview*

**Choose an install location with enough free space.** On shared systems (HPC, lab servers), home directories often have small quotas and inode limits that a full pixi environment (≈7–35 GB, 100k–350k files) will exhaust. Pick a path on a larger filesystem — a lab/project directory or scratch space. On a personal laptop, the default `$HOME/.pixi` is fine.

**Point pixi at that location** by temporarily overriding `HOME` and prepending pixi to `PATH` before running the installer. Replace `/your_pixi_install_path` with the directory you chose:

```bash
# Direct pixi to install into a location with enough space
export HOME=\"/your_pixi_install_path\"

# Make the pixi binary discoverable for this shell
export PATH=\"/your_pixi_install_path/.pixi/bin:$PATH\"
```

**On HPC clusters**, the installer can be memory-intensive — run it from an interactive compute node (for example, request ≥50 GB of memory) rather than the login node.

**Run the installer.** This downloads and installs pixi along with the StatFunGen-curated set of environments used by the xQTL protocol:

```bash
curl -fsSL https://raw.githubusercontent.com/StatFunGen/pixi-setup/refs/heads/main/pixi-setup.sh | bash
```

The installer will prompt you for the installation path and for an installation type (`minimal` vs. `full`). Choose `full` if you plan to run the complete xQTL protocol, since it includes TensorQTL, bcftools, plink, Seurat, and the Bioconductor packages the pipeline relies on.

After installation, restart your shell (or `source ~/.bashrc`) so that `pixi` is on your `PATH`.

To install additional packages later:

```bash
# Add a Python package
pixi global install -c conda-forge --environment python <package>

# Add an R package
pixi global install -c conda-forge --environment r-base r-<package>
```

### 3. Clone the xqtl-protocol repository

```bash
git clone https://github.com/StatFunGen/xqtl-protocol.git
cd xqtl-protocol
```

The `code/` directory contains the SoS notebooks that implement each mini-protocol. The `pipeline/` directory contains the underlying SoS workflows called by those notebooks.

### 4. Run an example pipeline

With the `sos` conda environment activated and `pixi` on your `PATH`, you can invoke any of the mini-protocols documented on this website. For example, to run TensorQTL on the toy dataset:

```bash
conda activate sos
sos run pipeline/TensorQTL.ipynb cis \\
    --genotype-file /path/to/genotype.bed \\
    --phenotype-file /path/to/phenotype.bed.gz \\
    --covariate-file /path/to/covariates.tsv \\
    --cwd output/
```

See the [Command Generator](https://statfungen.github.io/xqtl-protocol/code/commands_generator/eQTL_analysis_commands.html) for a ready-to-run set of commands that covers the full pipeline end-to-end."

## Analysis

Please visit [the homepage of the protocol website](https://statfungen.github.io/xqtl-protocol/) for the general background on this resource, in particular the [How to use the resource](https://statfungen.github.io/xqtl-protocol/README.html#how-to-use-the-resource) section. To perform a complete analysis from molecular phenotype quantification to xQTL discovery, please conduct your analysis in the order listed below, each link contains a mini-protocol for a specific task. All commands documented in each mini-protocol should be executed in the command line environment.

### Molecular Phenotype Quantification

Molecular phenotypic data is required for the generation of QTLs. We support bulk RNA-Seq, methylation and splicing phenotypes in our pipeline. Multiple [reference data](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data.html#) files are required before molecular phenotypes are quantified in samples. These include, but are not limited to, reference genomes, gene annotations, variant annotations, linkage disequilibirum data and topologically associated domains. [Quantification of gene expression](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/bulk_expression.html) is conducted with either RNA-SeQC for gene-level counts, or RSEM for transcript-level counts. [Quantification of alternative splicing events](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/splicing.html) is conducted with leafcutter2 to identify alternatively excised introns. [Quantification of DNA methylation](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/methylation.html) is done using SeSAMe. Each of these molecular phenotypes then undergo phenotype specific quality control and normalization.

### Data Pre-Processing

[Preprocessing of genotype data](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/genotype_preprocessing.html) begins with the application of variant filters using bcftools. VCF files are then converted to plink format so that kinship analyses may be performed to identify unrelated individuals. Genetic principal components are then generated for unrelated samples and genotype files are formatted for later generation of quantitative trait loci. 

[Preprocessing of phenotypic data](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype_preprocessing.html) begins with annotation of features, if required. Missing entries may then be imputed using a variety of methods included in the pipeline. Last, the phenotypes are formatted for later generation of quantitative trait loci. 

[Preprocessing of covariates](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/covariate_preprocessing.html) begins with the merging of phenotypic data with previously generated genetic principal components. The merged data is then used to calculate hidden factors which will later be used as additional covariates. 

### QTL Association Analysis

[QTL association analysis](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_testing.html) is conducted with TensorQTL. We include options for cis or trans analysis, with options to include interaction terms. [Hierarchical multiple testing](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_postprocessing.html) may then be applied to the results to adjust p-values. 

### Integrative Analysis

We include methods to conduct [TWAS](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html) in our pipeline to identify genes associated with complex traits. 

Our pipeline includes multiple methods for fine-mapping of QTLs. [Univariate fine-mapping and TWAS with SuSiE](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/univariate_fine_mapping_twas_vignette.html) generates TWAS weights and credible sets using SuSiE. [Regression with summary statistics](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/summary_stats_finemapping_vignette.html) allows for the inclusion of summary statistics from GWAS in SuSiE finemapping. [Univariate fine-mapping of functional data](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/univariate_fine_mapping_fsusie_vignette.html) uses epigenomic data to fine-map with fSuSiE. 

We also include method for [colocalization analysis](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/SuSiE_enloc.html). This starts with the generation of prior probabilities followed by pairwise colocalization analysis of xQTL and GWAS fine-mapping results to identifies shared causal variants. We also include an alternative method, [colocboost](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/colocboost.html), to identify shared genetic variants influencing multiple molecular traits. 

We utilize an [excess of overlap](https://statfungen.github.io/xqtl-protocol/code/enrichment/eoo_enrichment.html) method to evaluate the enrichment of significant variants within specific genomic annotations. [Pathway enrichment analysis](https://statfungen.github.io/xqtl-protocol/code/enrichment/gsea.html) identifies biological pathways that are statistically overrepresented in a given gene set, giving information on  potential biological functions, disease relevance, or regulatory mechanisms associated with the gene set. [Stratified LD Score Regression](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html) (S-LDSC) is used to quantify the contribution of different genomic functional annotations to the heritability of complex traits and assess their statistical significance. By integrating GWAS summary statistics with genome annotations, S-LDSC distinguishes true polygenic signals from confounding effects.





## Data

For record keeping: preparation of the demo dataset is documented [on this page](https://github.com/cumc/fungen-xqtl-analysis/tree/main/analysis/Wang_Columbia/ROSMAP/MWE) --- this is a private repository accessible to FunGen-xQTL analysis working group members.

For protocols listed in this page, downloaded required input data in [Synapse](https://www.synapse.org/#!Synapse:syn36416601). 
* To be able downloading the data, first create user account on [Synapse Login](https://www.synapse.org/). Username and password will be required when downloading
* Downloading required installing of Synapse API Clients, type `pip install synapseclient` in terminal or Command Prompt to install the Python package. Details list [on this page](https://help.synapse.org/docs/Installing-Synapse-API-Clients.1985249668.html).
* Each folder in different level has unique Synapse ID, which allowing you to download only some folders or files within the entire folder.

To download the test data for section "Bulk RNA-seq molecular phenotype quantification", please use the following Python codes,

```
import synapseclient 
import synapseutils 
syn = synapseclient.Synapse()
syn.login("your username on synapse.org","your password on synapse.org")
files = synapseutils.syncFromSynapse(syn, 'syn53174239', path="./")
```

To download the test data for section "xQTL association analysis", please use the following Python codes, 

```
import synapseclient 
import synapseutils 
syn = synapseclient.Synapse()
syn.login("your username on synapse.org","your password on synapse.org")
files = synapseutils.syncFromSynapse(syn, 'syn52369482', path="./")
```

## Software environment: pixi-managed environments

Analysis documented on this website uses software installed through [pixi](https://pixi.sh/) via the StatFunGen [`pixi-setup`](https://github.com/StatFunGen/pixi-setup) installer. See the *Step-by-step getting started guide* above for installation, and the Wang Lab documentation for more detail: <https://wanggroup.org/hpc/docs/software-setup-conda>.

Once `pixi` is installed and on your `PATH`, the pipelines will pick up the tools they need (TensorQTL, bcftools, plink, R packages, etc.) directly — there is no `--container` flag to pass and no Singularity/Docker image to download. If you previously used the `--container oras://ghcr.io/statfungen/...` option, it is no longer required.

### Troubleshooting

If you run into errors loading R libraries when the pipeline calls into R (for example, a locally installed R package shadowing the pixi-managed one), unset the user R library paths before invoking `sos`:

```bash
export R_LIBS=\"\"
export R_LIBS_USER=\"\"
```

For example, an error such as:

```
Error in dyn.load(file, DLLpath = DLLPath, ...):
unable to load shared object '$PATH/R/x86_64-pc-linux-gnu-library/4.2/stringi/libs/stringi.so':
libicui18n.so.63: cannot open shared object file: No such file or directory
```

is typically fixed by the two `export` statements above — they force R to use only the libraries provided by the pixi `r-base` environment.

If a required package is missing from the pixi environment, install it with:

```bash
# Python package
pixi global install -c conda-forge --environment python <package>

# R package
pixi global install -c conda-forge --environment r-base r-<package>
```

## Analyses on High Performance Computing clusters

The protocol example shown above performs analysis on a desktop workstation, as a demonstration. Typically the analyses should be performed on HPC cluster environments. This can be achieved via [SoS Remote Tasks](https://vatlab.github.io/sos-docs/doc/user_guide/task_statement.html) on [configured host computers](https://vatlab.github.io/sos-docs/doc/user_guide/host_setup.html). The mechanism is cluster-agnostic and works with SLURM, LSF, SGE, PBS/Torque, and other common schedulers. We provide a [toy example for running SoS pipeline on a typical HPC cluster environment](https://github.com/statfungen/xqtl-protocol/blob/main/code/misc/Job_Example.ipynb); first-time users are encouraged to try it out to help set up the computational environment necessary to run the analyses in this protocol."